# トークン重要度 / attention 分析の可視化

DP エンコーダのトークン構成(564 トークン)に対する **feature importance(除去テスト)** と
**fusion attention 分析** の結果を可視化するノートブック。

## ワークフロー

1. `scripts/token_importance.py` — トークン群除去(クラス drop / 近い順 top-K / 距離カットオフ)
   の FDE/ADE 影響を測定 → **TSV**
2. `scripts/attention_analysis.py --out_json ...` — fusion の attention シェア
   (クラス別 selectivity・層別・距離ビン・旋回条件付き)を集計 → **JSON**
3. このノートブックで両成果物を可視化

```bash
# 例(リポジトリルートで実行。MODEL_DIR は args.json + best_model.pth を含むディレクトリ)
MODEL_DIR=best_models/20260730/best_model
DATADIR=/path/to/mini_datasets/j6_2231_fullseq_mini_20260707
uv run python scripts/token_importance.py \
  --run_dir $MODEL_DIR --valid_set_list $DATADIR/path_list_valid.json \
  --n_samples 1024 --batch_size 64 --device cuda --out_tsv eval/token_importance.tsv
uv run python scripts/attention_analysis.py \
  --run_dir $MODEL_DIR --valid_set_list $DATADIR/path_list_valid.json \
  --n_samples 1024 --batch_size 64 --device cuda --out_json eval/attention.json
```

- **フルデータ検証**: `--valid_set_list` をフルデータセットの path list に差し替えるだけ
- ONNX しかない場合: `--run_dir` の代わりに `--onnx <path>`(args.json 同居前提)。
  ONNX では attention 分析は不可(内部重みへアクセスできないため)
- DDP 保存 checkpoint(`module.` プレフィックス)は自動対応済み
- スロット占有率は `uv run python scripts/token_occupancy_scan.py <dataset_root>`

背景・これまでの結果: `docs/token_imbalance_analysis.md`(未追跡の調査メモ)参照。

In [ ]:
from pathlib import Path

# ---- パス設定(環境に合わせて変更) ----
IMPORTANCE_TSV = Path("../best_models/20260730/eval/token_importance_n1024.tsv")
ATTENTION_JSON = Path("../best_models/20260730/eval/attention_analysis_n1024.json")

print(
    "importance:",
    IMPORTANCE_TSV,
    "->",
    "OK" if IMPORTANCE_TSV.exists() else "NOT FOUND (パスを設定してください)",
)
print("attention :", ATTENTION_JSON, "->", "OK" if ATTENTION_JSON.exists() else "NOT FOUND (任意)")

In [ ]:
import csv
import json

import matplotlib.pyplot as plt
import numpy as np

# 検証済みパレット(dataviz reference)。系列色はエンティティ固定
CAT = {"neighbors": "#2a78d6", "lanes": "#eb6834", "line_strings": "#1baf7a", "route": "#eda100"}
C_WORSE = "#eb6834"  # 劣化(+Δ)
C_BETTER = "#2a78d6"  # 改善(−Δ)
INK = "#1a1a19"
MUTED = "#6b6a63"
GRID = "#e5e4df"


def style(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.title.set_color(INK)
    return ax


rows = None
if IMPORTANCE_TSV.exists():
    with open(IMPORTANCE_TSV) as f:
        rows = [
            {k: (v if k == "config" else float(v)) for k, v in r.items()}
            for r in csv.DictReader(f, delimiter="\t")
        ]
    base = next(r for r in rows if r["config"] == "baseline")
    print(
        f"{len(rows)} configs, baseline fde_top={base['fde_top']:.2f}m ade_top={base['ade_top']:.2f}m"
    )

## 1. クラス除去の importance(permutation importance の token 版)

入力クラスを丸ごとゼロ埋め(= 正しい masking により attention から完全除去)したときの
top-mode FDE の変化。**+ が劣化(そのクラスをモデルが使っている)、− が改善(ノイズだった)**。

In [ ]:
if rows:
    drops = [r for r in rows if r["config"].startswith("drop:")]
    drops.sort(key=lambda r: r["d_fde_top"])
    names = [r["config"][5:] for r in drops]
    vals = [r["d_fde_top"] for r in drops]

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(drops) + 1))
    colors = [C_WORSE if v > 0 else C_BETTER for v in vals]
    ax.barh(names, vals, color=colors, height=0.62)
    ax.axvline(0, color=MUTED, lw=1)
    for i, v in enumerate(vals):
        ax.text(
            v + (0.03 if v >= 0 else -0.03),
            i,
            f"{v:+.2f}",
            va="center",
            ha="left" if v >= 0 else "right",
            fontsize=9,
            color=INK,
        )
    ax.set_xlabel("Δ top-mode FDE [m]  (+ = worse when removed)", color=MUTED)
    ax.set_title(f"Class-removal importance (baseline FDE {base['fde_top']:.2f} m)")
    ax.grid(axis="x", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

## 2. 近い順 top-K スイープ(トークン数最適化曲線)

近い K 要素だけ残して残りを除去。**曲線がベースラインに合流する最小 K が「必要トークン数」**。
スロット数の縮小や距離カットオフ設計の直接の根拠になる。

In [ ]:
if rows:
    sweeps = {
        "nbr_top": ("neighbors", 320),
        "lane_top": ("lanes", 140),
        "ls_top": ("line_strings", 60),
    }
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=False)
    for ax, (prefix, (cls, slots)) in zip(axes, sweeps.items()):
        pts = sorted(
            (
                (int(r["config"].split(":")[1]), r["fde_top"])
                for r in rows
                if r["config"].startswith(prefix + ":")
            ),
        )
        ks = [k for k, _ in pts] + [slots]
        ys = [y for _, y in pts] + [base["fde_top"]]
        ax.plot(ks, ys, marker="o", ms=5, lw=2, color=CAT[cls])
        ax.axhline(base["fde_top"], color=MUTED, lw=1, ls="--")
        ax.text(ks[0], base["fde_top"], " baseline", fontsize=8, color=MUTED, va="bottom")
        ax.set_xscale("log", base=2)
        ax.set_xticks(ks[:-1] + [slots])
        ax.set_xticklabels([str(k) for k in ks[:-1]] + [f"all\n({slots})"], fontsize=8)
        ax.set_title(f"{cls}: keep nearest K", fontsize=10)
        ax.set_xlabel("K (log scale)", color=MUTED)
        ax.grid(axis="y", color=GRID, lw=0.6)
        ax.set_axisbelow(True)
        style(ax)
    axes[0].set_ylabel("top-mode FDE [m]", color=MUTED)
    plt.tight_layout()
    plt.show()

    within = [r for r in rows if "_within:" in r["config"]]
    if within:
        print("距離カットオフ:")
        for r in within:
            print(f"  {r['config']:<16} fde={r['fde_top']:6.2f}  Δ={r['d_fde_top']:+.2f}")

## 3. attention シェア分析

`selectivity = attention シェア ÷ トークン数シェア`。**1.0 = 数比どおり(希釈)**、
>1 = 内容で選好、<1 = 抑制。「トークン数の多いクラスが attention を数で支配していないか」の検定。

In [ ]:
attn = None
if ATTENTION_JSON.exists():
    attn = json.loads(ATTENTION_JSON.read_text())
    print(f"n_samples={attn['n_samples']} (turning {attn['n_turning']}), layers={attn['n_layers']}")

if attn:
    classes = [
        c for c in attn["classes"] if attn["count_share"].get(c, 0) > 0 and c not in ("static",)
    ]
    sel = {c: attn["all_share_avg"][c] / attn["count_share"][c] for c in classes}
    order = sorted(classes, key=lambda c: sel[c])

    fig, ax = plt.subplots(figsize=(7, 0.45 * len(order) + 1))
    vals = [sel[c] for c in order]
    colors = [C_WORSE if v > 1 else C_BETTER for v in vals]
    ax.barh(order, vals, color=colors, height=0.62)
    ax.axvline(1.0, color=MUTED, lw=1)
    ax.text(1.0, len(order) - 0.2, " 1.0 = count-proportional (dilution)", fontsize=8, color=MUTED)
    for i, v in enumerate(vals):
        ax.text(v + 0.05, i, f"{v:.2f}x", va="center", fontsize=9, color=INK)
    ax.set_xlabel("selectivity (all-query attention share / token count share)", color=MUTED)
    ax.set_title("Attention selectivity by token class")
    ax.grid(axis="x", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
if attn:
    key_classes = [
        c for c in ("neighbors", "lanes", "line_strings", "route") if c in attn["classes"]
    ]
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    L = attn["n_layers"]
    for c in key_classes:
        ys = [100 * v for v in attn["ego_share_per_layer"][c]]
        ax.plot(range(L), ys, marker="o", ms=4, lw=2, color=CAT[c])
        ax.text(L - 1 + 0.08, ys[-1], c, fontsize=9, color=CAT[c], va="center")
    ax.set_xticks(range(L))
    ax.set_xticklabels([f"L{i}" for i in range(L)])
    ax.set_xlim(-0.3, L + 1.3)
    ax.set_xlabel("fusion layer", color=MUTED)
    ax.set_ylabel("ego-query attention share [%]", color=MUTED)
    ax.set_title("Ego-token attention share per fusion layer")
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

In [ ]:
if attn:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

    ax = axes[0]
    labels = ["turning", "straight"]
    vals = [100 * attn["route_share_turning"], 100 * attn["route_share_straight"]]
    ax.bar(labels, vals, color=CAT["route"], width=0.5)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.15, f"{v:.1f}%", ha="center", fontsize=10, color=INK)
    ax.set_ylabel("route attention share [%]", color=MUTED)
    ax.set_title("Route share: turning vs straight (ego query)")
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)

    ax = axes[1]
    bins = attn["dist_bins"]
    x = np.arange(len(bins))
    w = 0.38
    ax.bar(
        x - w / 2, [100 * v for v in attn["lane_bin_share"]], w, color=CAT["lanes"], label="lanes"
    )
    ax.bar(
        x + w / 2,
        [100 * v for v in attn["nbr_bin_share"]],
        w,
        color=CAT["neighbors"],
        label="neighbors",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(bins, fontsize=9)
    ax.set_ylabel("within-class attention share [%]", color=MUTED)
    ax.set_title("Attention by distance bin (ego query)")
    ax.legend(frameon=False, fontsize=9)
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    style(ax)
    plt.tight_layout()
    plt.show()

## 4. 読みどころ(これまでの結果との照合ポイント)

mini データでの 2 モデル(planTF V_tail / diffusion best 20260730)の結果からの予想。
フルデータモデルで以下がどう変わるかが検証ポイント:

1. **route パラドックス**: これまで route は attention を数比以上に受け(selectivity 1.3〜3.4x)、
   旋回時にシェアが増えるのに、除去しても FDE +0.2 程度しか劣化しない
   =「注目しているが中身が予測を変えない」。**2 モデル・2 デコーダで再現しており表現の問題と
   結論している。フルデータでも再現するか**が最重要の確認点
2. **必要トークン数**: これまで neighbors は top-16 で飽和、lanes は 40〜70 本で十分、
   line_strings は最寄り ~10 本に価値が集中。フルデータで K の合流点が右に動くか
3. **遠方カットオフ**: `*_within:100` はこれまで両モデルで無害〜改善
   (100m 超は正規化 10σ 超の外れ値領域)。フルデータでも無害なら導入根拠が固まる
4. **neighbors の逆反応**(diffusion best のみ): 近傍除去で開ループ FDE が改善(−0.97)。
   過剰な譲り学習か分布差かは未決着。**開ループ改善 ≠ 安全性改善**なので、
   数字が再現しても閉ループ評価なしに neighbor 削減を正当化しないこと

### 解釈上の注意

- 測っているのは「**その checkpoint が**何を使っているか」。劣化ゼロ=学習時から不要、ではない
  (最終確認は入力を削った構成での再学習比較)
- 部分除去(top-K)は学習分布外の入力パターンを作るため、クラス全除去より劣化が
  大きく出ることがある(V_tail の line_strings で観測)。importance はクラス全除去の値を採用する
- 距離ビンのシェアはトークン数正規化をしていない(ビン内トークン数の効果を含む)